<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<style>
h1 {
  font-size: 2.35rem !important;
  line-height: 1.15 !important;
  margin: 0.7rem 0 1rem !important;
}
h2 {
  font-size: 1.9rem !important;
  line-height: 1.2 !important;
  margin: 0.6rem 0 1rem !important;
}
h3 {
  font-size: 1.35rem !important;
  line-height: 1.3 !important;
  margin: 1.9rem 0 0.75rem !important;
}
h4 {
  font-size: 1.08rem !important;
  line-height: 1.35 !important;
  margin: 1.35rem 0 0.65rem !important;
}
.alchemi-api-label,
span[aria-label="ALCHEMI Toolkit API"] {
  color: currentColor !important;
  font-size: 0.70rem !important;
  font-weight: 700;
  letter-spacing: 0.05em;
}
@supports (color: color-mix(in srgb, black, white)) {
  .alchemi-api-label,
  span[aria-label="ALCHEMI Toolkit API"] {
    color: color-mix(in srgb, currentColor 68%, #76B900 32%) !important;
  }
}
</style>

# ALCHEMI Core · Module 02: Simulate and compose

**Goal:** Compose AIMNet2, dispersion, and electrostatics; run hot NCI trajectories; and keep weak complexes together with a bias hook.

**Estimated time:** 40 minutes  
**Next module:** [Module 03: Scale a study →](alchemi-core-03-scale-a-study.ipynb)

<img src="assets/course-roadmap-module02.svg" alt="ALCHEMI Core roadmap with Compose models highlighted in Module 02" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:1rem 0 0;">

Module 01 built `AtomicData` and `Batch`. Here, those same objects move through composed models and molecular dynamics.

In [ ]:
# Import the scientific Python, course-helper, and Toolkit APIs used below.
import random

import torch
from ase.units import kcal, mol
from IPython.display import display
from helpers import module02 as helpers
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import DynamicsStage, HostMemory, NVTLangevin
from nvalchemi.dynamics import initialize_velocities
from nvalchemi.dynamics.hooks import NaNDetectorHook, SnapshotHook
from nvalchemi.hooks import BiasedPotentialHook, DynamicsContext
from nvalchemi.models import AIMNet2Wrapper, DFTD3ModelWrapper, PipelineGroup
from nvalchemi.models import PipelineModelWrapper, PipelineStep
from nvalchemi.neighbors import compute_neighbors

In [ ]:
helpers.configure_tutorial()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
if device.type == "cuda":
    print("gpu:   ", torch.cuda.get_device_name(device))

## 1 · See how a well-composed Toolkit model reaches near-gold-standard accuracy

Intermolecular interactions drive molecular recognition, solvation, crystal packing, and adsorption. Even advanced MLIPs such as AIMNet2 benefit from explicit physical terms for these weak interactions.

Each panel follows one molecular pair as its separation changes. The curves show bare AIMNet2, AIMNet2 with D3 dispersion, and AIMNet2 with D3 plus charge-dependent electrostatics against CCSD(T)/CBS. DFT-D3 follows a similar trend and is omitted here to keep the comparison readable.

Each interaction energy is `E(AB) - E(A) - E(B)` in kcal/mol. More negative values mean stronger attraction. Across these 30 selected scan points, the complete route has `0.34 kcal/mol` MAE.

In [ ]:
# Load 90 frozen AB/A/B structures, aligned row metadata, and 30 references.
nci_atoms, nci_rows, nci_references = helpers.load_nci_atlas()
# Use the saved model curves for this result-first preview.
nci_curves = helpers.checked_nci_interaction_curves(nci_references)

{
    "scan points": len(nci_curves),
    "AB/A/B structures": len(nci_atoms)
}

The molecular geometries come from the [NCI Atlas](https://github.com/Honza-R/NCIAtlas), a high-quality collection of intermolecular interaction data.

`nci_rows` keeps each AB complex aligned with its isolated A and B fragments across the three scans.

The player moves through all three separation scans together. Each card shows the closest intermolecular distance `D min` and its change from equilibrium `ΔD`; the dotted line marks that atom pair.

In [ ]:
# Group the ten AB geometries for each complex and retain each fragment split.
scan_trajectories, scan_fragment_sizes, scan_scales = (
    helpers.nci_scan_player_data(nci_atoms, nci_rows)
)
# Build one synchronized player; its slider follows the shared source coordinate s.
helpers.show_nci_separation_player(
    scan_trajectories,
    scan_fragment_sizes,
    scan_scales,
)

In [ ]:
# Plot energy curves for the visualzied separations
helpers.plot_nci_interaction_curves(nci_curves, nci_references)

> **Bottom line**
>
> D3 and electrostatics are essential to capture intermolecualr interactions. Toolkit lets us compose both terms and pass the complete model into the same simulation API.

**🔗 Learn more in Toolkit docs**

- [Models and model composition](https://nvidia.github.io/nvalchemi-toolkit/userguide/models.html#composing-multiple-models) - Choose between additive and dependent composition.
- [AIMNet2 electrostatics pipeline](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/08_aimnet2_ewald_pipeline.html) - Follow predicted charges into electrostatics with autograd.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 2 · Compose the interaction model

We will rebuild all three curves from the same 90 frozen structures. As in Module 1, each ASE structure becomes one `AtomicData` object and `Batch` packs the independent structures on one device. Construction, execution, and inspection stay separate so each Toolkit action is visible.

In [ ]:
# Each of the 30 scan points supplies an AB complex and its A and B fragments.
# One Batch lets the model evaluate all 30 × 3 = 90 structures together in one forward pass on the selected device.
nci_scan_batch = Batch.from_data_list(
    [AtomicData.from_atoms(atoms, device=device) for atoms in nci_atoms],
    device=device,
)

{"molecular structures": nci_scan_batch.num_graphs, "atoms": nci_scan_batch.num_nodes}

### Load a built-in model wrapper

Toolkit includes built-in model wrappers, so supported pretrained models are ready to use with the same data and simulation APIs. A wrapper handles model-specific checkpoint loading and adapts native inputs and outputs to Toolkit's common model interface. The wrapped model accepts `AtomicData` and `Batch`, joins a model pipeline, and can drive Toolkit dynamics.

`AIMNet2Wrapper` is Toolkit's built-in wrapper for AIMNet2. Here it loads the pinned `aimnet2-wb97m-d3_0` molecular checkpoint directly onto the selected device and tensor dtype. `helpers.model_checkpoint()` resolves and checksum-verifies that file.

See the [AIMNet2 model card](https://huggingface.co/isayevlab/aimnet2-wb97m-d3) and [AIMNetCentral source and MIT license](https://github.com/isayevlab/aimnetcentral) for the checkpoint scope and reuse terms.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `AIMNet2Wrapper.from_checkpoint(...)`
>
> **Input:** an AIMNet2 checkpoint, device, and tensor dtype  
> **Returns:** a ready-to-call AIMNet2 model with Toolkit's common interface

Calling `.eval()` selects inference behavior in its PyTorch modules. Autograd remains available when a later calculation requests force derivatives.

In [ ]:
aimnet = AIMNet2Wrapper.from_checkpoint(
    helpers.model_checkpoint(),
    device=device,
    dtype=torch.float32,
).eval()

In [ ]:
# Update the wrapper configuration in place: one energy per structure and
# one predicted charge per atom. The next cell inspects the resolved interface.
aimnet.set_config(
    "active_outputs",
    {"energy", "charges"},
)

In [ ]:
# Inspect what the configured wrapper needs and returns before its first call.
{
    "required inputs": aimnet.input_data(),
    "active outputs": aimnet.model_config.active_outputs,
    "neighbor configuration": aimnet.model_config.neighbor_config,
    "molecules": nci_scan_batch.num_graphs,
    "atoms": nci_scan_batch.num_nodes,
}

### Build the model's neighbor list

AIMNet2 evaluates local atom pairs, so it needs a neighbor list in addition to positions and elements. `compute_neighbors(...)` writes those pairs into the `Batch` in place using the model's own `neighbor_config`. We prepare a clone so the shared scan batch remains available for the other model routes.

Calling the wrapper as `aimnet(batch)` returns a dictionary keyed by the requested outputs. This example reads the system-level `energy` and atom-level `charges` tensors.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `compute_neighbors(batch, config=model.model_config.neighbor_config)`
>
> **Input:** a `Batch` and the model's neighbor configuration  
> **Effect:** writes the neighbor list required by the next forward call

In [ ]:
aimnet_batch = nci_scan_batch.clone()
compute_neighbors(
    aimnet_batch,
    config=aimnet.model_config.neighbor_config,
)  # Writes AIMNet2's neighbor list into this clone.
aimnet_output = aimnet(aimnet_batch)

{
    "energy": tuple(aimnet_output["energy"].shape),
    "charges": tuple(aimnet_output["charges"].shape),
    "device": str(aimnet_output["energy"].device),
}

### Add explicit D3 dispersion

`DFTD3ModelWrapper` takes the AIMNet2 D3(BJ) coefficients `a1`, `a2`, and `s8`. Here `a1` and `s8` are dimensionless, `a2` is in Bohr, and `cutoff` is in Å.

`param_file` points to the D3 element-reference table installed with the course. The wrapper loads it when the model is created.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `DFTD3ModelWrapper(a1=..., a2=..., s8=..., cutoff=..., param_file=...)`
>
> **Input:** D3(BJ) damping coefficients, a neighbor cutoff, and the element-reference table  
> **Returns:** a Toolkit model wrapper that computes D3 energy and requested derivatives

In [ ]:
d3 = DFTD3ModelWrapper(
    a1=0.566,     # Dimensionless BJ damping coefficient
    a2=3.128,     # BJ damping length [Bohr]
    s8=0.3908,    # Dimensionless C8 scale
    cutoff=15.0,  # Neighbor cutoff [Å]
    param_file=helpers.d3_parameter_file(),
).to(device).eval()
# Request one system-level dispersion energy for every AB/A/B structure.
d3.set_config("active_outputs", {"energy"})

#### Run D3 on the same structures

The D3 wrapper has its own neighbor configuration. Build those neighbors, run one forward call, and inspect the system-level energy tensor.

In [ ]:
d3_batch = nci_scan_batch.clone()
compute_neighbors(d3_batch, config=d3.model_config.neighbor_config)
d3_output = d3(d3_batch)

{"D3 energy": tuple(d3_output["energy"].shape), "device": str(d3_output["energy"].device)}

### Add D3 as an independent pipeline group

A `PipelineModelWrapper` owns one or more `PipelineGroup` objects and exposes their combined outputs through Toolkit's common model interface. Each group below contains one independent model: AIMNet2 or D3. Its `steps` define what runs within that group, and `use_autograd=False` lets each model return its own energy for the wrapper to add.

`neighbor_adaptation="always"` builds one source neighbor list at the largest cutoff and filters it for the model with the smaller cutoff. After the forward call, we will subtract bare AIMNet2 and check that the combined energy added by this wrapper is the direct D3 energy.

In [ ]:
aimnet_d3 = PipelineModelWrapper(
    groups=[
        PipelineGroup(steps=[aimnet], use_autograd=False),
        PipelineGroup(steps=[d3], use_autograd=False),
    ],
    neighbor_adaptation="always",
).to(device).eval()
aimnet_d3.set_config("active_outputs", {"energy"})

In [ ]:
# Clone the common scans so this route owns its neighbor data.
aimnet_d3_batch = nci_scan_batch.clone()
compute_neighbors(
    aimnet_d3_batch,
    config=aimnet_d3.model_config.neighbor_config,
)  # One source list; the pipeline filters it for each model.
# Store the summed system energies for the identity check and interaction curve.
aimnet_d3_output = aimnet_d3(aimnet_d3_batch)

#### Check the energy added by the pipeline

The two groups are independent and their energies are added. Subtracting bare AIMNet2 from the pipeline result should therefore recover the direct D3 energy.

The next cell shows the D3 energy of the AB complex and its isolated A and B fragments for the first scan point, once from each route. It then reports the maximum absolute difference across all 90 structures. The following interaction plot evaluates model accuracy against CCSD(T)/CBS.

In [ ]:
# Subtract bare AIMNet2 to recover the D3 energy added by the pipeline.
pipeline_d3_energy = aimnet_d3_output["energy"] - aimnet_output["energy"]
direct_d3_energy = d3_output["energy"]

# The first three rows are the AB complex and its isolated A and B fragments.
pipeline_values = [round(float(value), 6) for value in pipeline_d3_energy[:3, 0]]
direct_values = [round(float(value), 6) for value in direct_d3_energy[:3, 0]]
print("order: AB complex, fragment A, fragment B")
print("pipeline D3 [eV]:", pipeline_values)
print("direct D3 [eV]:", direct_values)

max_d3_difference = (pipeline_d3_energy - direct_d3_energy).abs().max()
print(f"maximum absolute difference across 90 structures [eV]: {float(max_d3_difference):.6f}")

In [ ]:
# Convert total energies into interaction energies for the two model routes.
d3_curves = helpers.interaction_components(
    nci_rows,
    {"aimnet_base": aimnet_output["energy"], "base_d3": aimnet_d3_output["energy"]},
)

# Plot AIMNet2 + D3 beside bare AIMNet2 and the CCSD(T)/CBS reference.
helpers.plot_nci_interaction_curves(
    d3_curves, nci_references, components=("aimnet_base", "base_d3")
)

### Pass predicted charges between ordered steps

A `PipelineModelWrapper` organizes models into groups:

- `steps` inside one `PipelineGroup` run in order, so a later model can consume an earlier output.
- `groups` keep independent contributions and their derivative strategy separate.
- the pipeline sums energy and other additive outputs across the groups.

In the first group below, AIMNet2 predicts atomic `charges`. `wire={"charges": "partial_charges"}` writes that tensor onto the `Batch` under the input name required by the Coulomb step.

`helpers.DirectCoulombAdapter` is a course helper for finite, nonperiodic $1/r$ electrostatics. It uses those charges for atom pairs within 15 Å and returns energy and forces through Toolkit's model interface.

`use_autograd=True` sums the AIMNet2 and Coulomb energies first, then computes forces from the group energy. The derivative therefore includes the dependence of the Coulomb energy on AIMNet2's predicted charges. D3 stays in its own independent group.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `PipelineStep(aimnet, wire={"charges": "partial_charges"})`
>
> **Input:** AIMNet2's predicted `charges`  
> **Effect:** writes `partial_charges` for the following Coulomb step

In [ ]:
# Construct the finite, nonperiodic Coulomb term on the same device.
coulomb = helpers.DirectCoulombAdapter(
    cutoff=10.0,  # Finite electrostatic cutoff [Å]
).to(device).eval()
# Request energy now; forces are enabled before this model enters dynamics.
coulomb.set_config("active_outputs", {"energy"})

In [ ]:
# Run AIMNet2 before Coulomb and expose its charges as partial_charges.
interaction_model = PipelineModelWrapper(
    groups=[
        PipelineGroup(
            steps=[
                PipelineStep(aimnet, wire={"charges": "partial_charges"}),
                coulomb,
            ],
            use_autograd=True,  # Differentiate their summed, charge-dependent energy.
        ),
        # Add D3 as an independent energy and force contribution.
        PipelineGroup(steps=[d3], use_autograd=False),
    ],
    neighbor_adaptation="always",
).to(device).eval()
interaction_model.set_config("active_outputs", {"energy", "charges"})

In [ ]:
# Read the group configuration back from the constructed pipeline.
group_config = [
    {
        "models": [type(step.model).__name__ for step in group.steps],
        "wiring": [step.wire for step in group.steps],
        "use_autograd": group.use_autograd,
    }
    for group in interaction_model.groups
]

{
    "groups": group_config,
    "required batch inputs": interaction_model.input_data(),
    "active outputs": interaction_model.model_config.active_outputs,
}

In [ ]:
# Give the complete route its own Batch clone and shared source neighbors.
complete_batch = nci_scan_batch.clone()
compute_neighbors(
    complete_batch,
    config=interaction_model.model_config.neighbor_config,
)
# One forward call returns the composed energy and AIMNet2 atomic charges.
complete_output = interaction_model(complete_batch)

In [ ]:
# For each route, subtract E(A) and E(B) from E(AB), then convert eV
# to kcal/mol. The helper preserves the 30-point scan order from nci_rows.
computed_curves = helpers.interaction_components(
    nci_rows,
    {
        "aimnet_base": aimnet_output["energy"],
        "base_d3": aimnet_d3_output["energy"],
        "complete": complete_output["energy"],
    },
)

In [ ]:
# Compare all three live routes with CCSD(T)/CBS; legend values are MAEs.
helpers.plot_nci_interaction_curves(
    computed_curves,
    nci_references,
    components=("aimnet_base", "base_d3", "complete"),
)

In [ ]:
# Check whether the live calculation reproduces the saved Section 1 preview.
curve_deltas = helpers.nci_curve_deltas(computed_curves, nci_curves)
largest_curve_delta = max(curve_deltas.values())

# Check that atomic charges sum to each structure's known total charge.
charge_error = helpers.max_system_charge_error(
    nci_scan_batch, complete_output["charges"]
)

print(f"largest live/preview curve difference [kcal/mol]: {largest_curve_delta:.3f}")
print(f"largest system charge error [e]: {charge_error:.2e}")

### Test full NCIA250 dataset

The three scans above use 30 geometries to show how the model terms behave. NCIA250 broadens the accuracy check to 250 equilibrium complexes. The pinned AIMNet2 checkpoint supports 205 of them (sans inert gases), and `helpers.load_ncia250()` handles the routine archive parsing and element filtering.

Each supported interaction is stored as adjacent `AB`, `A`, and `B` structures. That gives 615 independent structures for one packed model call. We will reduce their energies to 205 interaction-energy predictions, then report one dataset-level error metric.

In [ ]:
# Load the 205 supported complexes as adjacent AB/A/B structures.
ncia_atoms, ncia_rows, ncia_references, _, _ = helpers.load_ncia250()

# Pack all 615 structures on the selected device for one model call.
ncia_batch = Batch.from_data_list(
    [AtomicData.from_atoms(atoms, device=device) for atoms in ncia_atoms],
    device=device,
)

In [ ]:
# Clone the Batch so this model call owns its neighbor data.
full_model_batch = ncia_batch.clone()
compute_neighbors(
    full_model_batch,
    config=interaction_model.model_config.neighbor_config,
)

# Evaluate all 615 AB/A/B structures together.
full_output = interaction_model(full_model_batch)
structure_energies_ev = full_output["energy"].squeeze(-1)

# Adjacent rows hold E(AB), E(A), and E(B) for each complex.
interaction_energies_ev = (
    structure_energies_ev[0::3]
    - structure_energies_ev[1::3]
    - structure_energies_ev[2::3]
)
predicted_interactions = (interaction_energies_ev * mol / kcal).detach().cpu()

print("full_output['energy'].shape:", tuple(full_output["energy"].shape))
print("first three interaction energies [kcal/mol]:", predicted_interactions[:3])

#### Try it: measure accuracy across 205 complexes

> ✏️ **TRY IT**
>
> The previous cell produced one interaction energy for each supported complex. Complete exactly two lines in the next cell:
>
> 1. select the 205 CCSD(T)/CBS reference interaction energies; and
> 2. average the absolute error of the 205 model predictions.
>
> **Success:** the check prints one MAE in kcal/mol.

In [ ]:
# TODO 1: select the 205 CCSD(T)/CBS interaction energies.
reference_energies = None

# TODO 2: average the absolute prediction error [kcal/mol].
mae_kcal_mol = None

if mae_kcal_mol is None:
    print("Complete the two TODO lines, then run this check again.")
else:
    print(f"MAE vs CCSD(T)/CBS: {float(mae_kcal_mol):.2f} kcal/mol")

> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

```python
reference_energies = torch.from_numpy(
    ncia_references["ccsd_t_cbs_kcal_mol"].to_numpy()
)
mae_kcal_mol = (
    predicted_interactions - reference_energies
).abs().mean()
```

The subtraction produces one prediction error for each of the 205 complexes. Taking the absolute value and then the mean gives one model-accuracy result for the full supported set.

</details>

**🔗 Learn more in Toolkit docs**

- [Composable model example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/07_composable_model_composition.html) - Build independent and dependent model groups.
- [D3 dispersion examples](https://nvidia.github.io/nvalchemi-toolkit-ops/examples/dispersion/index.html) - Explore batched GPU D3 energy and force calculations.

### ⚡ Power User: Does D3 still matter after electrostatics?

*Open this optional block to practice composing one new route from terms already used above.*

The complete route contains AIMNet2, electrostatics, and D3. Build a route with AIMNet2 and Coulomb, evaluate the same 90 structures, and compare its MAE with the complete route. The difference measures D3's value after electrostatics is already present.

#### Try it: compose the comparison route

> ✏️ **TRY IT**
>
> Fill only the two assignments in the next cell:
>
> 1. wrap AIMNet2 in `PipelineStep(...)` and wire `charges` to `partial_charges`;
> 2. put that step before `coulomb` in `comparison_steps`.
>
> **Hints:** use `{"charges": "partial_charges"}`. The supplied wrapper requests energy only, so its group uses `use_autograd=False`.
>
> Run the supplied evaluation and plot cells after filling the two lines.
>
> **Success:** the plot reports an MAE for AIMNet2 + electrostatics and for the complete route.

In [ ]:
# TODO 1: wire AIMNet2 charges to Coulomb's input name.
aimnet_charge_step = ...

# TODO 2: preserve the order AIMNet2 → Coulomb.
comparison_steps = ...

# The supplied wrapper exposes the two-step route as one energy model.
aimnet_coulomb = PipelineModelWrapper(
    groups=[PipelineGroup(steps=comparison_steps, use_autograd=False)],
    neighbor_adaptation="always",
).to(device).eval()
aimnet_coulomb.set_config("active_outputs", {"energy"})

In [ ]:
# Clone the same 90 structures and build neighbors for this comparison route.
aimnet_coulomb_batch = nci_scan_batch.clone()
compute_neighbors(
    aimnet_coulomb_batch,
    config=aimnet_coulomb.model_config.neighbor_config,
)
# Evaluate all structures together in one forward call.
aimnet_coulomb_output = aimnet_coulomb(aimnet_coulomb_batch)

# Convert both total-energy routes to aligned interaction energies [kcal/mol].
d3_value_curves = helpers.interaction_components(
    nci_rows,
    {
        "aimnet_coulomb": aimnet_coulomb_output["energy"],
        "complete": complete_output["energy"],
    },
)

In [ ]:
# Read D3's effect from the two MAEs in the legend.
helpers.plot_nci_interaction_curves(
    d3_value_curves,
    nci_references,
    components=("aimnet_coulomb", "complete"),
)

> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

```python
aimnet_charge_step = PipelineStep(
    aimnet,
    wire={"charges": "partial_charges"},
)
comparison_steps = [aimnet_charge_step, coulomb]
```

The first line transfers AIMNet2's predicted atomic charges to the input name expected by Coulomb. The second line fixes the execution order inside one `PipelineGroup`.

With the pinned checkpoint and these 30 scan points, the plot reports about `2.03 kcal/mol` MAE for AIMNet2 + electrostatics and `0.34 kcal/mol` for the complete route. D3 still contributes strongly after electrostatics is present.

</details>

Docs: [Composable model example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/07_composable_model_composition.html) · [AIMNet2 electrostatics pipeline](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/08_aimnet2_ewald_pipeline.html)

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 3 · Run molecular dynamics

<img src="assets/course-roadmap-module02-dynamics.svg" alt="ALCHEMI Core roadmap with Run dynamics highlighted in Module 02" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.75rem 0 0.5rem;">

Toolkit dynamics accepts the same model interface used for the interaction curves. We will first pass the complete model to `NVTLangevin` and run an ordinary trajectory.

The simulation reuses the three equilibrium complexes from the interaction scans. Later, we will clone the same initial state and add a bias so the two runs differ by one hook.

### Build the dynamics `Batch`

We will run the same three complexes from the interaction curves together in one dynamics `Batch`. The equilibrium geometry from each scan gives every trajectory a defined starting structure and supplies the reference separation (R_0) used by the later restraint.

Each AB structure stores all atoms from fragment A first, followed by fragment B. `fragment_a_sizes` records that boundary. Later cells use it to calculate the two fragment centers of mass and apply `BiasedPotentialHook` to their separation.

The helper handles routine NCI Atlas row selection. The next cell keeps the public `AtomicData.from_atoms(...)` and `Batch.from_data_list(...)` calls visible, including the energy and force fields that dynamics updates.

In [ ]:
# Select one equilibrium geometry per complex and record its fragment boundary.
md_labels, md_structures, fragment_a_sizes = helpers.equilibrium_nci_complexes(
    nci_atoms, nci_rows
)

[
    {
        "complex": label,
        "fragment A atoms": split,
        "fragment B atoms": len(structure) - split,
    }
    for label, structure, split in zip(
        md_labels, md_structures, fragment_a_sizes, strict=True
    )
]

In [ ]:
# Convert each ASE structure to one AtomicData record with masses and velocities.
md_records = []
for structure in md_structures:
    record = AtomicData.from_atoms(structure, device=device)
    # Dynamics writes one system energy and one three-vector force per atom.
    record.add_system_property("energy", torch.zeros((1, 1), device=device))
    record.add_node_property("forces", torch.zeros_like(record.positions))
    md_records.append(record)

# Pack the three heterogeneous complexes into the shared dynamics Batch.
md_start = Batch.from_data_list(md_records, device=device)

{
    "complexes": md_labels,
    "atoms per complex": md_start.num_nodes_per_graph.tolist(),
    "velocity shape": tuple(md_start.velocities.shape),
}

### Set up a plain Langevin trajectory

`NVTLangevin` integrates positions and velocities while a Langevin thermostat exchanges energy with each complex. This run uses:

- `temperature = 1,000 K` for a hot molecular stress test;
- `dt = 1.0 fs` for each integration step;
- `friction = 0.002 fs⁻¹`, corresponding to a 500 fs thermostat damping time;
- `n_steps = 3,000`, giving a 3.0 ps trajectory; and
- a fixed random seed so the later biased run can use the same velocities and noise sequence.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `NVTLangevin(model=..., dt=..., temperature=..., friction=..., n_steps=...)`
>
> **Input:** one Toolkit model and the thermostat controls above  
> **Returns:** a dynamics object that advances the complete `Batch`

`HostMemory(capacity=...)` allocates a CPU trajectory store. Its capacity counts individual systems, so every saved time point uses three records here.

### Prepare model outputs and one shared velocity sample

The interaction curves requested energies and charges. Molecular dynamics also needs one force vector per atom, so the next cell updates each model component's active outputs. The visible constants then define the thermostat and snapshot schedule.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `initialize_velocities(velocities=..., masses=..., temperature=..., batch_idx=..., random_seed=...)`
>
> **Input:** the existing velocity tensor, one mass and system index per atom, one target temperature per system, and a seed  
> **Effect:** writes a Maxwell-Boltzmann sample in place, removes center-of-mass drift, and rescales each system to its target temperature

`atomic_masses.reshape(-1)` supplies one scalar mass per atom. `batch_idx.int()` supplies the required 32-bit system index for every atom. We initialize once and clone this state so both trajectories start identically.

In [ ]:
# Dynamics needs system energies and atomic forces; charges still feed Coulomb.
aimnet.set_config("active_outputs", {"energy", "forces", "charges"})
d3.set_config("active_outputs", {"energy", "forces"})
coulomb.set_config("active_outputs", {"energy", "forces"})
interaction_model.set_config(
    "active_outputs", {"energy", "forces", "charges"}
)

In [ ]:
TEMPERATURE_K = 1000.0       # Thermostat target [K]
MD_TIMESTEP_FS = 1.0         # Integration timestep [fs]
MD_FRICTION_PER_FS = 0.002   # Langevin friction [fs⁻¹]
MD_STEPS = 3000              # 3.0 ps total trajectory
MD_RANDOM_SEED = 49          # Reproducible velocity and noise samples
SNAPSHOT_EVERY = 20          # Save every 20 integration steps

In [ ]:
# Give each system the same target temperature on the Batch device.
target_temperature = torch.full(
    (md_start.num_graphs,),
    TEMPERATURE_K,
    device=device,
    dtype=md_start.positions.dtype,
)

# Write one seeded thermal sample into the existing [atoms, 3] velocity tensor.
initialize_velocities(
    velocities=md_start.velocities,
    masses=md_start.atomic_masses.reshape(-1),
    temperature=target_temperature,
    batch_idx=md_start.batch_idx.int(),
    random_seed=MD_RANDOM_SEED,
)

In [ ]:
# Check the temperature realized for each system from its kinetic energy.
{"initial temperatures [K]": helpers.graph_temperatures(md_start).tolist()}

In [ ]:
# Include the initial state plus one saved state every 20 steps.
saved_time_points = 1 + (MD_STEPS + SNAPSHOT_EVERY - 1) // SNAPSHOT_EVERY
plain_store = HostMemory(capacity=saved_time_points * md_start.num_graphs)

# Build the ordinary 3.0 ps Langevin simulation.
plain_nvt = NVTLangevin(
    model=interaction_model,
    dt=MD_TIMESTEP_FS,
    temperature=TEMPERATURE_K,
    friction=MD_FRICTION_PER_FS,
    random_seed=MD_RANDOM_SEED,
    n_steps=MD_STEPS,
)

### Run vanilla trajectory

A **hook** attaches an action to a named point in the dynamics loop. Hooks can update model data, modify coordinates, energies, or forces, watch for numerical problems, and record trajectory frames. The dynamics algorithm stays in charge of when each action runs.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `plain_nvt.register_hook(hook, stage=DynamicsStage.BEFORE_COMPUTE)`
>
> **Input:** a hook and the dynamics stage where it should run  
> **Effect:** calls that hook at the selected point of every integration step

Here, `interaction_model.make_neighbor_hooks()` supplies model-owned hooks that rebuild each term's neighbor data as positions change. Registering them at `BEFORE_COMPUTE` updates those inputs immediately before the next model evaluation. `SnapshotHook` acts as a trajectory logger: it saves the current state every 20 steps, while the initialized state is stored once as \(t=0\).

Langevin first updates velocities using the force at the starting coordinates. The batch currently contains placeholder zeros, so `compute_neighbors(...)` and `compute(...)` evaluate the starting force before `run(...)`.

In [ ]:
# Clone the initialized state and save it as t = 0.
plain_start = md_start.clone()
plain_store.write(plain_start)

# Keep neighbor data current as positions change.
for neighbor_hook in interaction_model.make_neighbor_hooks():
    plain_nvt.register_hook(
        neighbor_hook, stage=DynamicsStage.BEFORE_COMPUTE
    )

# Save all three systems at the selected interval.
plain_nvt.register_hook(
    SnapshotHook(plain_store, frequency=SNAPSHOT_EVERY)
)

In [ ]:
# Evaluate the force used by the first velocity update.
compute_neighbors(
    plain_start,
    config=interaction_model.model_config.neighbor_config,
)
_ = plain_nvt.compute(plain_start)

In [ ]:
# Run the ordinary 3.0 ps trajectory.
plain_result = plain_nvt.run(plain_start)

# HostMemory saved three system records at each time point.
saved_plain_time_points = len(plain_store) // md_start.num_graphs
print("saved MD time points:", saved_plain_time_points)

### Keep a weak complex together during MD

<img src="assets/course-roadmap-module02-hooks.svg" alt="ALCHEMI Core roadmap with Modify with hooks highlighted in Module 02" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.75rem 0 0.5rem;">

The vanilla run uses the composed model directly. Intermolecular binding is weak enough that a hot trajectory can carry the two fragments apart. Some simulations instead need the complex to remain near its starting separation.

A hook adds that control at a named point in the dynamics step. Every hook has a `stage`, a `frequency`, and a `__call__(context, stage)` action. The live state is available as `context.batch`. Toolkit also provides hooks for snapshots, logging, NaN detection, energy-drift monitoring, force clamping, and periodic wrapping.

Here `BiasedPotentialHook` restrains the mass-weighted center-of-mass separation:

$$
R = \left\lVert \mathbf{r}_{\mathrm{COM},B} - \mathbf{r}_{\mathrm{COM},A} \right\rVert,
\qquad
\mathbf{r}_{\mathrm{COM}} =
\frac{\sum_i m_i \mathbf{r}_i}{\sum_i m_i}.
$$

The initial separation is $R_0$. The harmonic bias is

$$
U_{\mathrm{bias}}(R) = \frac{1}{2}k(R-R_0)^2,
\qquad
\left|F_{\mathrm{bias}}(R)\right| = k\left|R-R_0\right|.
$$

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `BiasedPotentialHook(fragment_restraint, stage=DynamicsStage.AFTER_COMPUTE)`
>
> **Input:** a callback returning one bias energy per system and one bias force per atom  
> **Effect:** adds those values to the model energy and forces before integration continues

The next cell measures one $R_0$ per complex. Then the callback uses $k=10\ \mathrm{eV\,\AA^{-2}}$, a stiff tutorial restraint chosen to make the 3 ps comparison easy to see.

In [ ]:
# Measure one initial fragment COM separation R0 [Å] per complex.
reference_separations = helpers.fragment_com_separations(
    md_start, fragment_a_sizes
)
{
    label: round(float(distance), 3)
    for label, distance in zip(md_labels, reference_separations, strict=True)
}

In [ ]:
# k controls how strongly each fragment separation returns to R0 [eV/Å²].
RESTRAINT_EV_PER_A2 = 10.0

def fragment_restraint(batch):
    """Return bias energy [systems, 1] and bias forces [atoms, 3]."""
    # AFTER_COMPUTE supplies current positions, masses, energy, and forces.
    return helpers.fragment_com_harmonic_bias(
        batch,
        fragment_a_sizes,
        reference_separations,
        RESTRAINT_EV_PER_A2,
    )

In [ ]:
# Allocate storage and match every plain-MD control.
biased_store = HostMemory(
    capacity=saved_time_points * md_start.num_graphs  # 151 times × 3 complexes
)
biased_nvt = NVTLangevin(
    model=interaction_model,          # Same composed potential
    dt=MD_TIMESTEP_FS,                # 1.0 fs
    temperature=TEMPERATURE_K,        # 1,000 K
    friction=MD_FRICTION_PER_FS,      # 0.002 fs⁻¹
    random_seed=MD_RANDOM_SEED,       # 49
    n_steps=MD_STEPS,                 # 3,000 steps = 3.0 ps
)

In [ ]:
# Clone the same initialized positions and velocities, then save t = 0.
biased_start = md_start.clone()
biased_store.write(biased_start)

# Reuse the same neighbor-update hooks as plain MD.
for neighbor_hook in interaction_model.make_neighbor_hooks():
    biased_nvt.register_hook(
        neighbor_hook, stage=DynamicsStage.BEFORE_COMPUTE
    )

In [ ]:
# Add the bias after each model force evaluation.
biased_nvt.register_hook(
    BiasedPotentialHook(
        fragment_restraint,
        stage=DynamicsStage.AFTER_COMPUTE,
    )
)

# Save the matched trajectory at the same interval.
biased_nvt.register_hook(
    SnapshotHook(biased_store, frequency=SNAPSHOT_EVERY)
)

In [ ]:
# Evaluate the model and bias at the shared starting coordinates.
compute_neighbors(
    biased_start,
    config=interaction_model.model_config.neighbor_config,
)
_ = biased_nvt.compute(biased_start)

In [ ]:
# Run the matched 3.0 ps trajectory with the bias hook active.
biased_result = biased_nvt.run(biased_start)
print("saved biased-MD time points:", len(biased_store) // md_start.num_graphs)

### Measure fragment separation over time

`SnapshotHook` writes each complex as a separate record. At every saved time point, the records appear in the same three-complex order used by `md_start`.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `flat_batch = plain_store.read()`
>
> **Input:** the snapshots accumulated in `HostMemory`  
> **Returns:** one flat `Batch` containing every saved system record

The next cell uses `to_data_list()` to expose those records and `Batch.from_data_list(...)` to repack each consecutive set of three as one synchronized frame. The following cell implements the mass-weighted COM equation above and produces a separation tensor with shape `[time points, complexes]`.

In [ ]:
# Repack each consecutive set of saved systems into one synchronized frame.
def regroup_saved_frames(store, systems_per_frame):
    # HostMemory.read() returns every saved system in write order as one Batch.
    flat_batch = store.read()
    # Split that Batch into one AtomicData record per saved system.
    records = flat_batch.to_data_list()
    return [
        # Each slice contains all systems saved at the same MD step.
        Batch.from_data_list(
            records[start : start + systems_per_frame],
            device=flat_batch.device,  # Keep frames on the storage device.
        )
        for start in range(0, len(records), systems_per_frame)
    ]


# The simulation advances three complexes together at every MD step.
systems_per_frame = md_start.num_graphs
# Apply the same grouping to the matched trajectories.
plain_md_frames = regroup_saved_frames(plain_store, systems_per_frame)
biased_md_frames = regroup_saved_frames(biased_store, systems_per_frame)

# Both lists use the same index for the same saved time point.
print("saved frames:", len(plain_md_frames))

#### Calculate fragment separation

Each saved frame is a `Batch` containing the same three complexes. For each system, `fragment_a_sizes` marks the first atom belonging to fragment B. The function below uses that boundary to calculate the mass-weighted center of each fragment, then returns one center-of-mass separation $R$ per complex in Å.

In [ ]:
def measure_fragment_separations(frame, fragment_splits):
    """Return one fragment COM separation per system [Å]."""
    separations = []
    for system_index, split in enumerate(fragment_splits):
        # Recover one complex from the synchronized frame.
        system = frame.get_data(system_index)
        masses = system.atomic_masses.reshape(-1, 1)  # [atoms, 1]
        positions = system.positions                 # [atoms, 3]

        # Atoms before and after split belong to fragments A and B.
        com_a = (masses[:split] * positions[:split]).sum(0) / masses[:split].sum()
        com_b = (masses[split:] * positions[split:]).sum(0) / masses[split:].sum()
        separations.append(torch.linalg.vector_norm(com_b - com_a))

    # Shape: [complexes]
    return torch.stack(separations)

In [ ]:
# Each call returns [complexes]; stacking frames adds the time dimension.
plain_separation_tensor = torch.stack([
    measure_fragment_separations(frame, fragment_a_sizes)
    for frame in plain_md_frames
])
biased_separation_tensor = torch.stack([
    measure_fragment_separations(frame, fragment_a_sizes)
    for frame in biased_md_frames
])

# Move these small [time points, complexes] analysis tensors to CPU.
plain_separation_tensor = plain_separation_tensor.cpu()
biased_separation_tensor = biased_separation_tensor.cpu()

print(
    "separation tensor [time points, complexes]:",
    tuple(plain_separation_tensor.shape),
)

In [ ]:
# Assign each tensor column to its complex label for plotting.
plain_separations = {
    label: plain_separation_tensor[:, index].tolist()
    for index, label in enumerate(md_labels)
}
biased_separations = {
    label: biased_separation_tensor[:, index].tolist()
    for index, label in enumerate(md_labels)
}

# Both dictionaries preserve the three-complex order.
print("separation series:", list(plain_separations))

In [ ]:
# Hook step 0 is the state after the first integration step, so add 1.
# Prepend the separately stored initial state at t = 0 fs.
time_fs = [0.0] + [
    (step + 1) * MD_TIMESTEP_FS
    for step in range(0, MD_STEPS, SNAPSHOT_EVERY)
]

In [ ]:
# Report one labeled row per complex; all distances are in Å.
for complex_index, label in enumerate(md_labels):
    r0 = float(reference_separations[complex_index])
    vanilla_trace = plain_separation_tensor[:, complex_index]
    biased_trace = biased_separation_tensor[:, complex_index]

    vanilla_max_change = float((vanilla_trace - r0).abs().max())
    biased_max_change = float((biased_trace - r0).abs().max())
    vanilla_final = float(vanilla_trace[-1])
    biased_final = float(biased_trace[-1])

    print(
        f"{label} [Å]: R0 {r0:.3f}; "
        f"vanilla max |R - R0| {vanilla_max_change:.3f}, final R {vanilla_final:.3f}; "
        f"biased max |R - R0| {biased_max_change:.3f}, final R {biased_final:.3f}"
    )

### Compare the matched trajectories

The summary reports $R_0$, the largest $|R-R_0|$, and the final $R$ for each complex. Read the saved output when browsing the notebook, or rerun the two trajectories to produce results for the selected device. The plot shows the complete 3 ps history.

Player shows vanilla MD on top and its bottom row shows the matched run with `BiasedPotentialHook`. One play button and one frame slider update all six structures.

Each card reports the current COM separation $R$. The biased row also reports the largest restraint-force magnitude reached so far:

$$
\max |F_{\mathrm{bias}}| = k\max |R-R_0|.
$$

Standard element colors show atom identity, and the dotted line joins the two fragment centers of mass.

In [ ]:
helpers.plot_fragment_separation_comparison(
    time_fs,
    plain_separations,
    biased_separations,
)

In [ ]:
# Build the shared-control comparison from the two saved trajectories.
bias_player = helpers.show_bias_comparison_player(
    plain_md_frames,
    biased_md_frames,
    labels=md_labels,
    fragment_sizes=fragment_a_sizes,
    reference_distances=reference_separations.tolist(),
    force_constant_ev_per_a2=RESTRAINT_EV_PER_A2,
)

In [ ]:
display(bias_player)

In [ ]:
# Confirm both simulations finished with finite coordinates.
print("plain MD positions are finite:", bool(torch.isfinite(plain_result.positions).all()))
print("biased MD positions are finite:", bool(torch.isfinite(biased_result.positions).all()))

### What changed

The two branches use the same model, starting coordinates, velocities, thermostat controls, and random sequence. The hook adds the harmonic fragment-separation energy and forces. The separation plot and six-card player show the resulting change for every complex.

> **Bottom line**
>
> `BiasedPotentialHook` changes the energy and forces seen by the integrator through a callback. The model and dynamics interfaces stay the same.

**🔗 Learn more in Toolkit docs**

- [Biased potential hook](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/01_biased_potential.html) - Add a bias through a dynamics hook.
- [NVT Langevin dynamics](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/05_nvt_langevin.html) - Configure temperature, timestep, friction, and reproducible noise.
- [Dynamics hooks](https://nvidia.github.io/nvalchemi-toolkit/userguide/hooks.html) - Register operations at named workflow stages.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

### ⚡ Power User: debug a broken MD update with hooks

*Imagine you are developing an MD algorithm. The coordinates begin evolving in an unexpected way, and a few steps later the model returns NaNs.*


The first debugging task is to locate the exact step where the trajectory becomes invalid and preserve that state for inspection. Once you have the failing coordinates, you can examine what changed and look for a pattern.

This short experiment makes that workflow reproducible. A small custom hook injects a coordinate collision at a seeded step. `SnapshotHook` records the updated coordinates before the model call, and `NaNDetectorHook` stops at the first non-finite energy or force.

`fault → snapshot → neighbor update → model compute → NaN detector`

The 20-step run is long enough to show normal updates before the injected failure.

#### Read a custom hook

A Toolkit hook is any object with three public pieces:

- `stage` selects the point in the workflow where it runs.
- `frequency` selects how often the engine calls it.
- `__call__(context, stage)` reads or updates the current workflow state.

For dynamics, `context.batch` is the live `Batch` and `context.step_count` is the current step. `context.model` and `context.workflow` are available when a hook needs them. Hooks that own a file, logger, or profiler can also implement `__enter__` and `__exit__` for setup and cleanup around `run()`.

The custom class below runs at `AFTER_PRE_UPDATE`, after the integrator moves the coordinates and before the model evaluates them.

In [ ]:
class CoordinateCollisionFaultHook:
    """Create one reproducible coordinate collision for this debug run."""

    stage = DynamicsStage.AFTER_PRE_UPDATE
    frequency = 1

    def __init__(self, system, pair, seed, trigger_window):
        self.system = system
        self.pair = pair
        self.trigger_step = random.Random(seed).randint(*trigger_window)

    def __call__(self, context: DynamicsContext, stage):
        if context.step_count == self.trigger_step:
            # batch_ptr marks where this system starts in the packed atom rows.
            start = int(context.batch.batch_ptr[self.system])
            # Convert the system-local pair into Batch row indices.
            first, second = (start + index for index in self.pair)
            # Mutate the live trajectory state outside PyTorch's gradient graph.
            with torch.no_grad():
                context.batch.positions[second].copy_(
                    context.batch.positions[first]
                )

In [ ]:
# Choose one system and local atom pair; the seed fixes a step in 12-18.
fault_hook = CoordinateCollisionFaultHook(
    system=1,
    pair=(0, 1),
    seed=23,
    trigger_window=(12, 18),
)

# Inspect the exact fault configuration before registering it.
{
    "system": md_labels[fault_hook.system],
    "local atom pair": fault_hook.pair,
    "trigger step": fault_hook.trigger_step,
}

#### Build the short debug workflow

This `NVTLangevin` instance runs at zero temperature and zero friction for 20 steps. The model already provides energy and forces from the preceding MD experiment. Here we create only a fresh trajectory store and a short integrator.

Its purpose is to advance the normal integration lifecycle far enough to reach the seeded fault. `HostMemory` reserves room for one saved state per system and step.

In [ ]:
# Reserve one pre-compute snapshot per system for at most 20 debug steps.
debug_store = HostMemory(capacity=20 * md_start.num_graphs)
# A short 0.2 fs step advances the lifecycle; T=0 and friction=0 add no noise.
debug_nvt = NVTLangevin(
    model=interaction_model,
    dt=0.2,
    temperature=0.0,
    friction=0.0,
    random_seed=23,
    n_steps=20,
)

#### Register the diagnostic chain

The stages follow the data that we need to inspect:

1. The fault hook runs at `AFTER_PRE_UPDATE`, after coordinates move.
2. `SnapshotHook` runs next at the same stage and stores those coordinates before the model call. Hooks at one stage run in registration order.
3. The model's neighbor hooks run at `BEFORE_COMPUTE` and rebuild neighbor data for the changed coordinates.
4. `NaNDetectorHook` runs at `AFTER_COMPUTE`. It checks `energy` and `forces` with `torch.isfinite` and raises a `RuntimeError` containing the step, fields, counts, and affected system indices.

`NaNDetectorHook(extra_keys=[...])` can inspect additional batch tensors. Its `frequency` can reduce monitoring cost in a long production run.

In [ ]:
debug_nvt.register_hook(fault_hook)
debug_nvt.register_hook(
    SnapshotHook(
        debug_store,
        stage=DynamicsStage.AFTER_PRE_UPDATE,
    )
)

for neighbor_hook in interaction_model.make_neighbor_hooks():
    debug_nvt.register_hook(
        neighbor_hook, stage=DynamicsStage.BEFORE_COMPUTE
    )
debug_nvt.register_hook(
    NaNDetectorHook(stage=DynamicsStage.AFTER_COMPUTE)
)

#### Run until the detector stops the trajectory

The integrator's first velocity update needs the force at the clean starting coordinates. We prepare the model inputs and call `compute(...)` once before `run(...)`.

The coordinate hook then creates the exact overlap at its selected step. The expected `RuntimeError` is the result of this debugging experiment, and its message reports the affected fields and systems. The following cell reads `HostMemory` directly and selects the saved system at that same step.

In [ ]:
debug_start = md_start.clone()
compute_neighbors(
    debug_start,
    config=interaction_model.model_config.neighbor_config,
)
debug_nvt.compute(debug_start)

try:
    debug_nvt.run(debug_start)
except RuntimeError as error:
    # Keep the notebook running and show NaNDetectorHook's full report.
    print(error)

In [ ]:
# HostMemory contains three system records for every completed pre-update.
debug_records = debug_store.read().to_data_list()
systems_per_step = md_start.num_graphs
captured_time_points = len(debug_records) // systems_per_step

# Select the injected system from the saved records at the exact trigger step.
fault_record = fault_hook.trigger_step * systems_per_step + fault_hook.system
captured = debug_records[fault_record]
first, second = fault_hook.pair
captured_distance = torch.linalg.vector_norm(
    captured.positions[second] - captured.positions[first]
)
nonfinite_positions = int((~torch.isfinite(captured.positions)).sum())

print("captured time points:", captured_time_points)
print(f"faulty pair distance [Å]: {float(captured_distance):.3f}")
print("non-finite position values:", nonfinite_positions)

> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the failure interpretation</strong></summary>

The saved coordinates remain finite and contain the injected \(0.0\ \mathrm{\AA}\) pair. The following model call produces non-finite energy or forces, and `NaNDetectorHook` stops the run. `SnapshotHook` preserves the exact input state that caused the failure.

</details>

Docs: [Safety and monitoring](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/05_safety_and_monitoring.html) · [`NaNDetectorHook` API](https://nvidia.github.io/nvalchemi-toolkit/modules/dynamics/generated/nvalchemi.dynamics.hooks.NaNDetectorHook.html) · [Custom hook example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/02_custom_hook.html)

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

### ⚡ Power User: implement Velocity Verlet with `BaseDynamics`

*Open this optional block when you want to add an integration rule while keeping Toolkit's model and hook lifecycle.*

Velocity Verlet advances each atom $i$ from step $n$ to $n+1$. Its position, velocity, force, and mass are $\mathbf r_i^n$, $\mathbf v_i^n$, $\mathbf F_i^n$, and $m_i$.

$$
\begin{aligned}
\mathbf v_i^{n+1/2}
  &= \mathbf v_i^n + \frac{\Delta t}{2}\frac{\mathbf F_i^n}{m_i}
  && \text{first half-kick} \\
\mathbf r_i^{n+1}
  &= \mathbf r_i^n + \Delta t\,\mathbf v_i^{n+1/2}
  && \text{position drift} \\
\mathbf F_i^{n+1}
  &= -\nabla_{\mathbf r_i} E(\mathbf r^{n+1})
  && \text{model evaluation} \\
\mathbf v_i^{n+1}
  &= \mathbf v_i^{n+1/2} + \frac{\Delta t}{2}\frac{\mathbf F_i^{n+1}}{m_i}
  && \text{second half-kick}
\end{aligned}
$$

#### Map the equations to Toolkit

One `BaseDynamics` step calls the three methods in this order:

1. `pre_update(batch)` applies the first half-kick with $\mathbf F^n$, then updates the positions.
2. `compute(batch)` evaluates the model at $\mathbf r^{n+1}$ and writes $\mathbf F^{n+1}$.
3. `post_update(batch)` applies the second half-kick with the new force.

`atomic_masses.reshape(-1, 1)` gives one mass per atom and broadcasts it across the three Cartesian components. Multiplying a timestep in femtoseconds by `ase.units.fs` converts it to the units used by these tensor updates.

`__needs_keys__ = {"forces"}` requires the model to return forces. `__provides_keys__ = {"positions", "velocities"}` declares the fields that this integrator updates.

#### Try it: translate Velocity Verlet into tensor updates

> ✏️ **TRY IT**
>
> Complete the three TODOs in `StudentVelocityVerlet`:
>
> 1. In `pre_update`, add $\frac{\Delta t}{2}\mathbf F^n/m$ to `batch.velocities`.
> 2. Still in `pre_update`, add $\Delta t\,\mathbf v^{n+1/2}$ to `batch.positions`.
> 3. In `post_update`, add $\frac{\Delta t}{2}\mathbf F^{n+1}/m$ to `batch.velocities`.
>
> `BaseDynamics` evaluates the model between the two methods. Run the next comparison with Toolkit's built-in `NVE` after filling the TODOs.
>
> **Success:** the printed position and velocity differences remain close to numerical precision across all four steps.

In [ ]:
from ase.units import fs as ASE_FS
from nvalchemi.dynamics import BaseDynamics


class StudentVelocityVerlet(BaseDynamics):
    __needs_keys__ = {"forces"}
    __provides_keys__ = {"positions", "velocities"}

    def __init__(self, model, dt, **kwargs):
        super().__init__(model=model, **kwargs)
        self.dt = dt * ASE_FS  # Convert femtoseconds to the update units.

    def pre_update(self, batch):
        masses = batch.atomic_masses.reshape(-1, 1)
        # Update the live trajectory state outside PyTorch's gradient graph.
        with torch.no_grad():
            # Δv = (Δt / 2) F(t) / m
            # TODO 1: update batch.velocities in place.

            # Δr = Δt v(t + Δt / 2)
            # TODO 2: update batch.positions in place.
            pass

    def post_update(self, batch):
        masses = batch.atomic_masses.reshape(-1, 1)
        # Apply the second state update with the refreshed model force.
        with torch.no_grad():
            # Δv = (Δt / 2) F(t + Δt) / m
            # TODO 3: update batch.velocities in place.
            pass

#### Compare each update with Toolkit's `NVE`

`NVE` is Toolkit's built-in Velocity Verlet integrator. The next four cells prepare identical state, construct both integrators, evaluate the shared starting force, and compare four paired `step(...)` updates.

Each output row reports the largest position and velocity difference after one update. Read the values directly to see how closely the two implementations agree.

In [ ]:
from nvalchemi.dynamics import NVE

VALIDATION_TIMESTEP_FS = 0.25  # Integration timestep [fs]
VALIDATION_STEPS = 4           # Number of paired updates

# Select one complex, then give each integrator an independent clone.
validation_start = md_start[1:2]
custom_batch = validation_start.clone()
toolkit_batch = validation_start.clone()

In [ ]:
# Construct the custom and built-in integrators with identical controls.
custom_vv = StudentVelocityVerlet(
    interaction_model,
    dt=VALIDATION_TIMESTEP_FS,
    hooks=interaction_model.make_neighbor_hooks(),
)
toolkit_nve = NVE(
    interaction_model,
    dt=VALIDATION_TIMESTEP_FS,
    hooks=interaction_model.make_neighbor_hooks(),
)

In [ ]:
# Both first half-kicks must use F(t) at the same starting coordinates.
for dynamics, batch in (
    (custom_vv, custom_batch),
    (toolkit_nve, toolkit_batch),
):
    compute_neighbors(
        batch,
        config=interaction_model.model_config.neighbor_config,
    )
    dynamics.compute(batch)

In [ ]:
# Advance both public integrators once, then compare their updated states.
for step in range(1, VALIDATION_STEPS + 1):
    custom_vv.step(custom_batch)
    toolkit_nve.step(toolkit_batch)

    max_position_difference = (
        custom_batch.positions - toolkit_batch.positions
    ).abs().max()
    max_velocity_difference = (
        custom_batch.velocities - toolkit_batch.velocities
    ).abs().max()

    print(
        f"step {step}: max |Δr| {float(max_position_difference):.3e} Å; "
        f"max |Δv| {float(max_velocity_difference):.3e}"
    )


> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

```python
class StudentVelocityVerlet(BaseDynamics):
    __needs_keys__ = {"forces"}
    __provides_keys__ = {"positions", "velocities"}

    def __init__(self, model, dt, **kwargs):
        super().__init__(model=model, **kwargs)
        self.dt = dt * ASE_FS

    def pre_update(self, batch):
        masses = batch.atomic_masses.reshape(-1, 1)
        with torch.no_grad():
            batch.velocities.add_(
                0.5 * self.dt * batch.forces / masses
            )
            batch.positions.add_(self.dt * batch.velocities)

    def post_update(self, batch):
        masses = batch.atomic_masses.reshape(-1, 1)
        with torch.no_grad():
            batch.velocities.add_(
                0.5 * self.dt * batch.forces / masses
            )
```

The first method sees $\mathbf F(t)$. The model call between the methods refreshes the force, so the second method sees $\mathbf F(t + \Delta t)$.

The comparison calls `step(...)` on both public integrators from identical initial state. Each output row reports the largest coordinate difference `max |Δr|` and velocity difference `max |Δv|` after one paired update. The printed values show how closely the learner's tensor updates reproduce Toolkit's built-in `NVE` across all four steps.

</details>

Docs: [Custom integrator example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/05_custom_integrator.html) · [Integrator algorithms](https://nvidia.github.io/nvalchemi-toolkit/modules/dynamics/methods.html) · [NVE energy-conservation example](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/04_nve_energy_conservation.html)

### Learn about Toolkit

Continue with three official examples that extend the main ideas in this module:

- [Composable model composition](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/07_composable_model_composition.html) - Build independent model terms and ordered property transfers with `PipelineModelWrapper`.
- [Custom hook](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/02_custom_hook.html) - Read workflow state and accumulate a structural observable at a chosen stage.
- [Custom integrator](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/05_custom_integrator.html) - Implement a new update rule with `BaseDynamics`.

## Module 02 recap

You composed AIMNet2, D3, and electrostatics through Toolkit interface into one capable model. You passed that model directly to Langevin dynamics, then used `BiasedPotentialHook` to keep weak molecular complexes near their starting separation. The optional Power User blocks extend the same APIs to emulate and debug debug failure and a custom Velocity Verlet integrator.

Module 03 scales the same model into multistage computational campaign in Toolkit. 

**Next module:** [Module 03: Scale a study →](alchemi-core-03-scale-a-study.ipynb)